# 🔥 Allan QLoRA — Шаг 2: Дообучение (запускать повторно)

**Этот ноутбук запускается для каждого чанка датасета.**

При каждом запуске он:
1. Читает `progress.json` с Google Drive
2. Определяет, какой датасет и чанк ещё не обучен
3. Загружает только этот кусок данных
4. Запускает QLoRA-дообучение
5. Сохраняет LoRA-адаптер на Drive
6. Обновляет `progress.json` — следующий запуск возьмёт **следующий** чанк

---
**Перед запуском:** убедитесь что выполнили `01_qlora_ru_setup.ipynb`

## Шаг 0: Установка библиотек и монтирование Drive

In [ ]:
# Установка (быстро, если уже установлено)
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.11.1 \
    bitsandbytes==0.43.3 accelerate==0.33.0 datasets==3.0.1 \
    sentencepiece protobuf einops

# Монтирование Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive подключён")

## Шаг 1: Конфигурация обучения

Здесь можно выбрать датасет вручную или оставить `AUTO` — тогда автоматически берётся следующий необученный чанк.

In [ ]:
import json
from pathlib import Path

# ================================================================
# НАСТРОЙКА: Измените при необходимости
# ================================================================

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

# Выбор датасета:
# 'AUTO'                — автоматически берёт следующий необученный чанк
# 'ru_turbo_alpaca'     — Русский Alpaca (~40k инструкций)
# 'ru_turbo_saiga'      — Saiga диалоги (~40k)
# 'oasst_ru'            — OpenAssistant RU (~8k)
# 'russian_instructions'— Большой инструкционный датасет (~150k)
# 'ru_wikipedia'        — Русская Википедия (очень большой!)
# 'ru_news'             — Новости Gazeta.ru (~63k)
DATASET_NAME = 'AUTO'

# QLoRA параметры (оптимизированы для T4 15GB)
QLORA_CONFIG = {
    'lora_r': 16,            # ранг LoRA (16-64, выше = больше VRAM)
    'lora_alpha': 32,        # масштаб LoRA (обычно = 2*r)
    'lora_dropout': 0.05,
    'target_modules': 'auto', # 'auto' определит сам по архитектуре
}

# Параметры тренировки
TRAIN_CONFIG = {
    'num_train_epochs': 1,        # эпох на чанк (1 достаточно для последовательного обучения)
    'per_device_train_batch_size': 2,  # для T4 с 7B моделью
    'gradient_accumulation_steps': 8,  # эффективный батч = 16
    'learning_rate': 2e-4,
    'max_seq_length': 512,        # максимальная длина последовательности
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'fp16': True,                 # T4 поддерживает fp16
    'bf16': False,                # bf16 только на A100/H100
    'save_steps': 100,
    'logging_steps': 10,
    'optim': 'paged_adamw_8bit',  # экономия памяти
    'gradient_checkpointing': True,
}

# ================================================================

# Загрузка конфига
config_path = DRIVE_ROOT / 'config.json'
if not config_path.exists():
    raise FileNotFoundError(f"❌ config.json не найден: {config_path}\nСначала запустите 01_qlora_ru_setup.ipynb")

with open(config_path) as f:
    project_config = json.load(f)

BASE_MODEL = project_config.get('base_model', 'Qwen/Qwen2.5-7B-Instruct')
print(f"✅ Конфиг загружен")
print(f"   Базовая модель: {BASE_MODEL}")
print(f"   Корень проекта: {DRIVE_ROOT}")

## Шаг 2: Определение следующего чанка

In [ ]:
import json
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
progress_path = DRIVE_ROOT / 'progress.json'

if not progress_path.exists():
    raise FileNotFoundError(f"❌ progress.json не найден. Сначала запустите 01_qlora_ru_setup.ipynb")

with open(progress_path, 'r', encoding='utf-8') as f:
    progress = json.load(f)

# Определяем датасет и чанк
selected_dataset = None
selected_chunk_idx = None

if DATASET_NAME == 'AUTO':
    # Автоматически находим первый датасет с незавершёнными чанками
    for ds_name, info in progress.items():
        completed = set(info['completed_chunks'])
        total = info['total_chunks']
        # Ищем первый незавершённый чанк
        for chunk_idx in range(total):
            if chunk_idx not in completed:
                selected_dataset = ds_name
                selected_chunk_idx = chunk_idx
                break
        if selected_dataset:
            break
    
    if selected_dataset is None:
        print("🎉 ВСЕ ЧАНКИ ВСЕХ ДАТАСЕТОВ ЗАВЕРШЕНЫ!")
        print("   Переходите к 03_qlora_ru_merge_export.ipynb")
        raise SystemExit("Обучение завершено для всех датасетов.")
else:
    # Выбран конкретный датасет
    if DATASET_NAME not in progress:
        raise ValueError(f"❌ Датасет '{DATASET_NAME}' не найден в progress.json")
    
    info = progress[DATASET_NAME]
    completed = set(info['completed_chunks'])
    
    # Берём следующий незавершённый чанк
    selected_dataset = DATASET_NAME
    selected_chunk_idx = None
    for chunk_idx in range(info['total_chunks']):
        if chunk_idx not in completed:
            selected_chunk_idx = chunk_idx
            break
    
    if selected_chunk_idx is None:
        print(f"✅ Датасет '{DATASET_NAME}' полностью обучен!")
        raise SystemExit(f"Датасет {DATASET_NAME} завершён.")

# Выводим информацию о выбранном чанке
ds_info = progress[selected_dataset]
chunk_size = ds_info['chunk_size']
total_chunks = ds_info['total_chunks']
chunk_start = selected_chunk_idx * chunk_size
chunk_end = min(chunk_start + chunk_size, ds_info['approx_total'])

print("=" * 60)
print("ПЛАН ТЕКУЩЕЙ СЕССИИ ОБУЧЕНИЯ")
print("=" * 60)
print(f"Датасет:    {selected_dataset}")
print(f"Описание:   {ds_info['description']}")
print(f"Чанк:       {selected_chunk_idx + 1} из {total_chunks}")
print(f"Строки:     {chunk_start} — {chunk_end} ({chunk_size} примеров)")
print(f"Завершено:  {len(ds_info['completed_chunks'])}/{total_chunks} чанков")
print("=" * 60)

# Путь для сохранения адаптера
adapter_save_path = (
    DRIVE_ROOT / 'adapters' /
    f"{selected_dataset}_chunk{selected_chunk_idx:04d}"
)
print(f"\nАдаптер сохранится в: {adapter_save_path}")

## Шаг 3: Загрузка и подготовка чанка датасета

In [ ]:
from datasets import load_dataset, Dataset
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
ds_info = progress[selected_dataset]

# Конфигурация датасетов (дублируем из setup для независимости ноутбука)
DATASETS_CONFIG = {
    "ru_turbo_alpaca": {
        "hf_name": "IlyaGusev/ru_turbo_alpaca",
        "split": "train",
        "subset": None,
        "format": "alpaca",
    },
    "ru_turbo_saiga": {
        "hf_name": "IlyaGusev/ru_turbo_saiga",
        "split": "train",
        "subset": None,
        "format": "saiga",
    },
    "oasst_ru": {
        "hf_name": "IlyaGusev/oasst_ru",
        "split": "train",
        "subset": None,
        "text_column": "text",
        "format": "text",
    },
    "russian_instructions": {
        "hf_name": "Den4ikAI/russian_instructions",
        "split": "train",
        "subset": None,
        "format": "alpaca",
    },
    "ru_wikipedia": {
        "hf_name": "wikimedia/wikipedia",
        "split": "train",
        "subset": "20231101.ru",
        "text_column": "text",
        "format": "text",
    },
    "ru_news": {
        "hf_name": "IlyaGusev/gazeta",
        "split": "train",
        "subset": None,
        "text_column": "text",
        "format": "text",
    },
}

ds_cfg = DATASETS_CONFIG[selected_dataset]
hf_name = ds_cfg['hf_name']
subset = ds_cfg.get('subset')
split = ds_cfg.get('split', 'train')
fmt = ds_cfg['format']

print(f"📥 Загрузка датасета: {hf_name}")
print(f"   Чанк {selected_chunk_idx}: строки {chunk_start}—{chunk_end}")
print("   (Используется streaming для экономии RAM)\n")

# Загружаем через streaming и берём только нужный чанк
if subset:
    raw_stream = load_dataset(hf_name, subset, split=split,
                              streaming=True, trust_remote_code=True)
else:
    raw_stream = load_dataset(hf_name, split=split,
                              streaming=True, trust_remote_code=True)

# Берём строго нужный диапазон, пропускаем уже обученные
raw_samples = []
for i, sample in enumerate(raw_stream):
    if i < chunk_start:
        continue
    if i >= chunk_end:
        break
    raw_samples.append(sample)
    if (i - chunk_start + 1) % 500 == 0:
        print(f"  Загружено: {i - chunk_start + 1}/{chunk_size} примеров")

print(f"\n✅ Загружено примеров: {len(raw_samples)}")

In [ ]:
# Форматирование датасета в единый текстовый формат для обучения

def format_alpaca(sample):
    """Формат: Alpaca (instruction, input, output)"""
    instruction = sample.get('instruction', '').strip()
    inp = sample.get('input', '').strip()
    output = sample.get('output', '').strip()
    
    if inp:
        text = f"### Задание:\n{instruction}\n\n### Контекст:\n{inp}\n\n### Ответ:\n{output}"
    else:
        text = f"### Задание:\n{instruction}\n\n### Ответ:\n{output}"
    return text


def format_saiga(sample):
    """Формат: Saiga (messages list)"""
    messages = sample.get('messages', [])
    if not messages:
        # Попытка с другим ключом
        prompt = sample.get('prompt', '')
        response = sample.get('response', '')
        return f"Пользователь: {prompt}\nАссистент: {response}"
    
    parts = []
    role_map = {'user': 'Пользователь', 'assistant': 'Ассистент', 'system': 'Система'}
    for msg in messages:
        role = role_map.get(msg.get('role', 'user'), 'Пользователь')
        content = msg.get('content', '').strip()
        if content:
            parts.append(f"{role}: {content}")
    return '\n'.join(parts)


def format_text(sample, text_col='text'):
    """Формат: plain text"""
    # Пробуем разные ключи
    for key in [text_col, 'text', 'content', 'article']:
        val = sample.get(key, '')
        if val:
            return val.strip()
    return ''


# Форматируем все примеры
print("🔧 Форматирование данных...")
formatted_texts = []
skipped = 0

for sample in raw_samples:
    if fmt == 'alpaca':
        text = format_alpaca(sample)
    elif fmt == 'saiga':
        text = format_saiga(sample)
    elif fmt == 'text':
        text_col = ds_cfg.get('text_column', 'text')
        text = format_text(sample, text_col)
    else:
        text = str(sample)
    
    # Фильтрация: пропускаем слишком короткие тексты
    if len(text.strip()) < 20:
        skipped += 1
        continue
    formatted_texts.append({'text': text})

print(f"✅ Отформатировано: {len(formatted_texts)} примеров (пропущено коротких: {skipped})")

# Показываем пример
if formatted_texts:
    print("\n--- Пример (первые 300 символов) ---")
    print(formatted_texts[0]['text'][:300])
    print("---")

# Создаём HuggingFace Dataset из списка
train_dataset = Dataset.from_list(formatted_texts)
print(f"\n✅ Dataset создан: {len(train_dataset)} примеров")

## Шаг 4: Загрузка модели с QLoRA (4-bit квантизация)

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

# Определяем: грузить с Drive (кэш) или напрямую с HuggingFace
model_cache = DRIVE_ROOT / 'models' / BASE_MODEL.replace('/', '_')
model_source = str(model_cache) if (model_cache / 'config.json').exists() else BASE_MODEL
print(f"📥 Источник модели: {model_source}")

# --- 4-bit квантизация (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NormalFloat4 — лучший для LLM
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # двойная квантизация = меньше VRAM
)

print(f"\n🔧 Загрузка токенизатора {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side='right',
)

# Устанавливаем pad_token если отсутствует
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("   pad_token = eos_token")

print(f"   Vocab size: {tokenizer.vocab_size}")

print(f"\n🔧 Загрузка модели в 4-bit (QLoRA)...")
print(f"   VRAM до загрузки: {torch.cuda.memory_allocated()/1024**3:.2f} ГБ")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

print(f"   VRAM после загрузки: {torch.cuda.memory_allocated()/1024**3:.2f} ГБ")
print(f"✅ Модель загружена")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Подготовка модели для QLoRA-обучения
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=TRAIN_CONFIG['gradient_checkpointing']
)

# Автоопределение target_modules по архитектуре модели
def get_target_modules(model):
    """Определяет модули для LoRA по типу архитектуры."""
    model_type = model.config.model_type.lower()
    
    # Qwen2 / Qwen2.5
    if 'qwen2' in model_type:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    # LLaMA / LLaMA-3 / Mistral / Saiga
    elif model_type in ['llama', 'mistral', 'mixtral']:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    # Phi-3
    elif 'phi' in model_type:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_up_proj', 'down_proj']
    # Gemma
    elif 'gemma' in model_type:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    # Fallback: ищем Linear-слои с нужными именами
    else:
        import re
        modules = set()
        for name, module in model.named_modules():
            if any(k in name for k in ['q_proj', 'k_proj', 'v_proj', 'query', 'key', 'value']):
                modules.add(name.split('.')[-1])
        return list(modules) if modules else ['q_proj', 'v_proj']

if QLORA_CONFIG['target_modules'] == 'auto':
    target_modules = get_target_modules(model)
else:
    target_modules = QLORA_CONFIG['target_modules']

print(f"🎯 Target modules (LoRA): {target_modules}")

# Применяем LoRA
lora_config = LoraConfig(
    r=QLORA_CONFIG['lora_r'],
    lora_alpha=QLORA_CONFIG['lora_alpha'],
    lora_dropout=QLORA_CONFIG['lora_dropout'],
    target_modules=target_modules,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n✅ LoRA-адаптер подключён")
print(f"   VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} ГБ")

## Шаг 5: Обучение

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from datetime import datetime
from pathlib import Path
import json
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

# Папка для чекпоинтов текущей сессии
session_id = datetime.now().strftime('%Y%m%d_%H%M')
checkpoint_dir = (
    DRIVE_ROOT / 'checkpoints' /
    f"{selected_dataset}_chunk{selected_chunk_idx:04d}_{session_id}"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# TensorBoard логи
log_dir = DRIVE_ROOT / 'logs' / f"{selected_dataset}_chunk{selected_chunk_idx:04d}_{session_id}"
log_dir.mkdir(parents=True, exist_ok=True)

print(f"📝 Чекпоинты: {checkpoint_dir}")
print(f"📊 Логи: {log_dir}")
print(f"\n🚀 Запуск обучения...")
print(f"   Датасет: {selected_dataset}, чанк {selected_chunk_idx}")
print(f"   Примеров: {len(train_dataset)}")
print(f"   Эпох: {TRAIN_CONFIG['num_train_epochs']}")
print(f"   Эффективный batch: {TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']}")
print()

# Конфигурация обучения через SFTConfig (trl >= 0.9)
training_args = SFTConfig(
    output_dir=str(checkpoint_dir),
    num_train_epochs=TRAIN_CONFIG['num_train_epochs'],
    per_device_train_batch_size=TRAIN_CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=TRAIN_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAIN_CONFIG['learning_rate'],
    fp16=TRAIN_CONFIG['fp16'],
    bf16=TRAIN_CONFIG['bf16'],
    optim=TRAIN_CONFIG['optim'],
    save_steps=TRAIN_CONFIG['save_steps'],
    logging_steps=TRAIN_CONFIG['logging_steps'],
    warmup_ratio=TRAIN_CONFIG['warmup_ratio'],
    lr_scheduler_type=TRAIN_CONFIG['lr_scheduler_type'],
    max_seq_length=TRAIN_CONFIG['max_seq_length'],
    dataset_text_field='text',
    report_to='tensorboard',
    logging_dir=str(log_dir),
    save_total_limit=2,              # держать только 2 последних чекпоинта
    dataloader_num_workers=2,
    packing=False,                   # False = надёжнее для коротких текстов
    gradient_checkpointing=TRAIN_CONFIG['gradient_checkpointing'],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# ОБУЧЕНИЕ
train_result = trainer.train()

print("\n✅ Обучение завершено!")
print(f"   Loss: {train_result.training_loss:.4f}")
print(f"   Шагов: {train_result.global_step}")

## Шаг 6: Сохранение адаптера и обновление прогресса

In [ ]:
import json
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
progress_path = DRIVE_ROOT / 'progress.json'

# Сохраняем LoRA-адаптер
adapter_save_path = (
    DRIVE_ROOT / 'adapters' /
    f"{selected_dataset}_chunk{selected_chunk_idx:04d}"
)
adapter_save_path.mkdir(parents=True, exist_ok=True)

print(f"💾 Сохранение LoRA-адаптера в {adapter_save_path}...")
model.save_pretrained(str(adapter_save_path))
tokenizer.save_pretrained(str(adapter_save_path))
print(f"✅ Адаптер сохранён")

# Проверяем размер
import subprocess
result = subprocess.run(['du', '-sh', str(adapter_save_path)], capture_output=True, text=True)
print(f"   Размер: {result.stdout.split()[0]}")

# Обновляем progress.json
with open(progress_path, 'r', encoding='utf-8') as f:
    progress = json.load(f)

ds_progress = progress[selected_dataset]

# Добавляем чанк в завершённые
if selected_chunk_idx not in ds_progress['completed_chunks']:
    ds_progress['completed_chunks'].append(selected_chunk_idx)
    ds_progress['completed_chunks'].sort()

# Находим следующий незавершённый чанк
completed_set = set(ds_progress['completed_chunks'])
next_chunk = None
for i in range(ds_progress['total_chunks']):
    if i not in completed_set:
        next_chunk = i
        break

ds_progress['next_chunk'] = next_chunk  # None если всё завершено
ds_progress['adapter_paths'][str(selected_chunk_idx)] = str(adapter_save_path)

# Сохраняем историю сессии
session_record = {
    'chunk_idx': selected_chunk_idx,
    'chunk_start': chunk_start,
    'chunk_end': chunk_end,
    'training_loss': train_result.training_loss,
    'steps': train_result.global_step,
    'adapter_path': str(adapter_save_path),
    'timestamp': datetime.now().isoformat(),
}
ds_progress['training_sessions'].append(session_record)

# Сохраняем обновлённый progress.json
with open(progress_path, 'w', encoding='utf-8') as f:
    json.dump(progress, f, ensure_ascii=False, indent=2)

print(f"\n✅ progress.json обновлён")
print(f"\n=" * 60)
print("ИТОГ СЕССИИ")
print("=" * 60)
print(f"Датасет: {selected_dataset}")
print(f"Обучен чанк: {selected_chunk_idx} ({chunk_start}—{chunk_end})")
print(f"Loss: {train_result.training_loss:.4f}")
print(f"Адаптер: {adapter_save_path}")

completed_now = len(ds_progress['completed_chunks'])
total_chunks = ds_progress['total_chunks']
print(f"\nПрогресс датасета: {completed_now}/{total_chunks} чанков")

if next_chunk is not None:
    next_start = next_chunk * ds_progress['chunk_size']
    next_end = min(next_start + ds_progress['chunk_size'], ds_progress['approx_total'])
    print(f"\n➡️  СЛЕДУЮЩИЙ ЗАПУСК возьмёт чанк {next_chunk} (строки {next_start}—{next_end})")
    print(f"   Просто запустите этот ноутбук снова!")
else:
    print(f"\n🎉 Датасет '{selected_dataset}' ПОЛНОСТЬЮ ОБУЧЕН!")
    print(f"   При следующем AUTO-запуске перейдёт к следующему датасету.")

## Шаг 7: Быстрая проверка качества модели

In [ ]:
# Тест модели после обучения
import torch

model.eval()

test_prompts = [
    "### Задание:\nОбъясни что такое машинное обучение простыми словами.\n\n### Ответ:\n",
    "### Задание:\nНапиши короткое стихотворение о зиме на русском языке.\n\n### Ответ:\n",
    "### Задание:\nКак приготовить борщ? Дай краткий рецепт.\n\n### Ответ:\n",
]

print("=" * 60)
print("ТЕСТ МОДЕЛИ")
print("=" * 60)

for i, prompt in enumerate(test_prompts):
    print(f"\n--- Тест {i+1} ---")
    print(f"Запрос: {prompt.split(chr(10))[1]}")
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    print(f"Ответ: {generated[:400]}")

print("\n✅ Тест завершён")
print("\n💡 Следующий шаг: запустите этот ноутбук снова для обучения следующего чанка.")
print("   Когда все нужные датасеты обучены → запускайте 03_qlora_ru_merge_export.ipynb")

## Просмотр общего прогресса (можно запускать в любой момент)

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
progress_path = DRIVE_ROOT / 'progress.json'

with open(progress_path) as f:
    progress = json.load(f)

print("=" * 60)
print("ОБЩИЙ ПРОГРЕСС ОБУЧЕНИЯ")
print("=" * 60)

total_done = 0
total_all = 0

for ds_name, info in progress.items():
    done = len(info['completed_chunks'])
    total = info['total_chunks']
    total_done += done
    total_all += total
    
    # Прогресс-бар
    bar_len = 20
    filled = int(bar_len * done / total) if total > 0 else 0
    bar = '█' * filled + '░' * (bar_len - filled)
    pct = 100 * done / total if total > 0 else 0
    
    status = "✅" if done == total else ("🔄" if done > 0 else "⏳")
    next_c = info.get('next_chunk')
    
    print(f"\n{status} {ds_name}")
    print(f"   {info['description']}")
    print(f"   [{bar}] {done}/{total} чанков ({pct:.0f}%)")
    if next_c is not None:
        next_start = next_c * info['chunk_size']
        print(f"   Следующий: чанк {next_c} (с примера {next_start})")
    
    if info['training_sessions']:
        last = info['training_sessions'][-1]
        print(f"   Последнее обучение: {last['timestamp'][:16]}, loss={last['training_loss']:.4f}")

print(f"\n{'=':=<60}")
bar_len = 30
filled = int(bar_len * total_done / total_all) if total_all > 0 else 0
bar = '█' * filled + '░' * (bar_len - filled)
pct = 100 * total_done / total_all if total_all > 0 else 0
print(f"ИТОГО: [{bar}] {total_done}/{total_all} ({pct:.1f}%)")